# 01-9_remove_wcs_in_HDUL_ys

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [2]:
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

This notebook was generated at 2023-02-01 18:57:31 (KST = GMT+0900) 
0 Python     3.8.16 64bit [GCC 11.2.0]
1 IPython    8.8.0
2 OS         Linux 5.15.0 58 generic x86_64 with glibc2.17
3 numpy      1.22.3
4 pandas     1.5.2
5 matplotlib 3.6.3
6 scipy      1.8.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 version_information 1.0.4


### import modules

In [3]:
import os, shutil
from glob import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.stats import sigma_clip
from astropy.io import fits
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

from snuo1mpy import Preprocessor

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

/home/guitar79/Downloads/ysphotutilpy/ysphotutilpy/seputil.py:112: UserWarning: Package sep is not installed. Some functions will not work.
  warn("Package sep is not installed. Some functions will not work.")


In [6]:
#%%
BASEDIR = astro_utilities.base_dir

BASEDIRs = sorted(Python_utilities.getFullnameListOfsubDir(BASEDIR))
print ("BASEDIRs: {}".format(BASEDIRs))
print ("len(BASEDIRs): {}".format(len(BASEDIRs)))


BASEDIR = Path(BASEDIRs[0])
print ("Starting...\n{}".format(BASEDIR))

BASEDIR = Path(BASEDIR)

RESULTDIR = BASEDIR / astro_utilities.DAOfinder_result_dir
SOLVEDDIR = BASEDIR / astro_utilities.solved_dir
MASTERDIR = BASEDIR / astro_utilities.master_dir
REDUCEDDIR = BASEDIR / astro_utilities.reduced_dir
MASTERDIR = BASEDIR / astro_utilities.master_dir

if not MASTERDIR.exists():
    os.makedirs("{}".format(str(MASTERDIR)))
    print("{} is created...".format(str(MASTERDIR)))

BASEDIRs: ['/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-24_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-25_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-27_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-02_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-04_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-11-17_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/LANDOLT-114670_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/', '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/LANDOLT-114670_Light_-_2022-10-24_-_RiLA600_

In [7]:

summary = yfu.make_summary(BASEDIR/"*.fit*")
#print(summary)
print("len(summary):", len(summary))
print("summary:", summary)
#print(summary["file"][0])

All 31 keywords (guessed from /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/-_Bias_-_2022-11-15-12-01-37_000sec_RiLA600_STX-16803_-19C_2bin.fit) will be loaded.


/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key EXTEND not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/-_Flat_B_2022-10-25-08-44-49_000sec_RiLA600_STX-16803_-25C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key DATE-LOC not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/-_Flat_B_2022-10-25-08-44-49_000sec_RiLA600_STX-16803_-25C_2bin.fit, filling with None.
  warn(str_keyerror_fill.format(k, str(item)))
/home/guitar79/Downloads/ysfitsutilpy/ysfitsutilpy/filemgmt.py:298: UserWarning: Key FOCRATIO not found for /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/-_Flat_B_2022-10-25-08-44-49_000sec_RiLA600_STX-16803_-25C_2bin.fit, filling with None.

len(summary): 304
summary:                                                   file  filesize  SIMPLE  \
0    /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8392320    True   
1    /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8392320    True   
2    /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8392320    True   
3    /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8392320    True   
4    /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8392320    True   
..                                                 ...       ...     ...   
299  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
300  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
301  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
302  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
303  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   

     BITPIX  NAXIS  NAXIS1  NAXIS2 EXTEND    BZERO IMAGETYP 

/tmp/ipykernel_21046/2584850535.py:4: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  print("summary:", summary)


In [8]:
df_light = summary[summary["IMAGETYP"] == "LIGHT"]
print ("df_light: {}".format(df_light))
print ("len(df_light): {}".format(len(df_light)))

df_light:                                                   file  filesize  SIMPLE  \
172  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
173  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
174  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
175  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
176  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
..                                                 ...       ...     ...   
299  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
300  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
301  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
302  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   
303  /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_...   8395200    True   

     BITPIX  NAXIS  NAXIS1  NAXIS2 EXTEND    BZERO IMAGETYP  ...  FOCALLEN  \

In [16]:
for _, row in df_light.iterrows():
        # 파일명 출력
        print (row["file"])
        fpath = Path(row["file"])
        new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_clean.fit")
        # fits hedaer 에 있는 wcs 정보를 지운다
        yfu.wcsremove(fpath, 
                    additional_keys=["COMMENT"],
                    verbose=True,
                    output=new_fpath,
                    ccddata=True,
                    overwrite=True)
        if new_fpath.exists() \
            and fpath.exists():
            print("rename", f"{str(new_fpath)}", f"{str(fpath)}")
            #os.rename(f"{str(new_fpath)}", f"{str(fpath)}")
            shutil.move(f"{str(new_fpath)}", f"{str(fpath)}")
            
    # except Exception as err :
    #     print("X"*60)
    #     print('{0}'.format(err))

/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin.fit
Removed keywords: 
rename /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit /mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin.fit
/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit
Removed keywords: 


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/Rdata/CCD_obs/RnE_2022/RiLA600_STX-16803_2bin/KLEOPATRA_Light_-_2022-10-23_-_RiLA600_STX-16803_-_2bin/KLEOPATRA_Light_b_2022-10-23-11-59-11_180sec_RiLA600_STX-16803_-30C_2bin_clean.fit'